# Build Your Own "Golden Gate Claude"!

### A fun, hands-on walkthrough of SAE feature steering with nanochat

---

**What was Golden Gate Claude?**

In May 2024, Anthropic's interpretability team made a discovery that captured the internet's imagination. Using Sparse Autoencoders (SAEs), they found a single **feature** inside Claude — one direction in the model's internal representation space — that responded to the concept of the **Golden Gate Bridge**.

When they artificially **amplified** that feature during inference, something remarkable happened: Claude became *obsessed* with the Golden Gate Bridge. Ask about math? *"Well, the Golden Gate Bridge has some lovely mathematical curves..."* Ask about feelings? *"I feel like the Golden Gate Bridge on a foggy morning..."*

It was funny, viral, and profoundly important. It demonstrated that:

1. **SAEs find human-interpretable concepts** inside neural networks
2. **Amplifying a single feature** can dramatically reshape model behavior
3. **We can steer AI systems** at a level more precise than prompting

**What we'll do in this notebook:**

We'll recreate the exact same technique on a nanochat model — going from zero to your own "Golden Gate" steered model in under 10 minutes!

| Step | What | Why |
|------|------|-----|
| 1 | Build a small GPT | Our miniature "Claude" |
| 2 | Collect activations | Record the model's "thoughts" |
| 3 | Train an SAE | Decompose thoughts into features |
| 4 | Find interesting features | Discover our "Golden Gate" |
| 5 | **Steer the model!** | Crank up a feature and watch behavior change |
| 6 | Visualize everything | Interactive dashboards |

**No expensive GPU required** — this runs on Colab's free tier (CPU or T4)!

## Setup

First, let's clone the repo and install dependencies. This takes about 1-2 minutes on Colab.

In [ ]:
# Check if we're on Colab
import os
ON_COLAB = 'COLAB_GPU' in os.environ or 'GOOGLE_COLAB' in os.environ or os.path.exists('/content')

if ON_COLAB:
    %cd /content
    !git clone https://github.com/SolshineCode/nanochat-SAE.git 2>/dev/null || echo "Already cloned"
    %cd nanochat-SAE

    # Install Rust toolchain (needed for the BPE tokenizer)
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>/dev/null
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

    # Build the Rust BPE tokenizer
    !cd rustbpe && pip install -e . 2>/dev/null

    # Install other deps
    !pip install torch numpy tqdm -q
    print("Setup complete!")
else:
    print("Running locally -- make sure you're in the nanochat-SAE directory")

In [ ]:
import torch
import torch.nn.functional as F
import time
import sys
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Make sure we can import nanochat modules
sys.path.insert(0, str(Path(".").resolve()))

from nanochat.gpt import GPT, GPTConfig
from sae.config import SAEConfig
from sae.models import TopKSAE
from sae.hooks import ActivationCollector
from sae.trainer import SAETrainer
from sae.evaluator import SAEEvaluator
from sae.feature_viz import FeatureVisualizer
from sae.runtime import InterpretableModel

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print("All imports successful!")

## Step 1: Build the Brain

Every language model is a transformer — a stack of layers that progressively refine a "residual stream" (a vector that encodes the model's evolving understanding of the input).

Real Claude has billions of parameters. Our nanochat model is tiny, but it has the **same architecture**: attention heads, MLP layers, residual streams. Think of it as a toy brain we can fully inspect.

```
Input tokens → [Layer 0] → [Layer 1] → ... → [Layer 7] → Output predictions
                   ↑                              ↑
              "early thoughts"              "refined thoughts"
              (syntax, basic)               (semantic, complex)
```

In [ ]:
# Build our miniature transformer
model_config = GPTConfig(
    sequence_len=128,   # context window (128 tokens)
    vocab_size=512,     # small vocabulary
    n_layer=8,          # 8 transformer layers
    n_head=4,           # 4 attention heads per layer
    n_kv_head=4,        # key/value heads (same as query heads here)
    n_embd=128,         # 128-dimensional hidden state ("thought vector")
)

model = GPT(model_config)
model.init_weights()
# Re-randomize output head (init_weights zeros it for training stability)
torch.nn.init.normal_(model.lm_head.weight, std=0.02)
model = model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model architecture: {model_config.n_layer} layers, {model_config.n_head} heads, d={model_config.n_embd}")
print(f"Total parameters: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"\nThis is our miniature 'Claude' — same transformer architecture, much smaller scale!")

## Step 2: Record the Brain's Thoughts

When a transformer processes text, each layer produces a **residual stream** — a vector that encodes everything the model has figured out so far. We'll tap into **layer 4** (the middle of our 8-layer model) and record thousands of these thought-vectors.

This is like putting an **EEG on our model's brain**!

> **Why the middle layer?** Early layers handle syntax and basic patterns. Late layers handle high-level semantics. The middle layers are often where the most interesting "concept" features live — and it's where Anthropic found the Golden Gate Bridge feature in Claude.

In [ ]:
# Choose which layer to tap into
target_layer = 4  # Middle of our 8-layer model
hook_point = f"blocks.{target_layer}.hook_resid_post"

# Set up the activation collector (like attaching electrodes to the brain)
collector = ActivationCollector(
    model=model,
    hook_points=[hook_point],
    max_activations=15_000,  # Collect 15K thought-vectors
    device="cpu",            # Store on CPU to save GPU memory
)

# Feed data through the model and record activations
t0 = time.time()
with torch.no_grad(), collector:
    for i in range(30):
        tokens = torch.randint(0, model_config.vocab_size, (8, model_config.sequence_len), device=device)
        model(tokens)
        if (i + 1) % 10 == 0:
            print(f"  Batch {i+1}/30 — {collector.counts[hook_point]:,} thought-vectors recorded")

activations = collector.get_activations()[hook_point]
elapsed = time.time() - t0

print(f"\nCollected {activations.shape[0]:,} thought-vectors of dimension {activations.shape[1]}")
print(f"Time: {elapsed:.1f}s")
print(f"\nEach thought-vector is a {activations.shape[1]}-dimensional point in 'thought space'")

## Step 3: Train the SAE — Decomposing Thoughts into Concepts

Here's the **key insight** behind Golden Gate Claude:

A 128-dimensional thought-vector is **dense** — every dimension is active at once. It's like hearing 128 instruments all playing simultaneously. Hard to understand!

A **Sparse Autoencoder** learns to decompose each thought into a combination of just a **few features** from a much larger dictionary:

```
Dense thought (128 dims, all active)
        ↓ [SAE Encoder]
Sparse features (1024 dims, only 16 active!)
        ↓ [SAE Decoder]
Reconstructed thought (128 dims) ≈ original
```

Each feature in the SAE dictionary corresponds to a **concept** — in a trained model, these might be things like "bridge", "math", "negation", "happy", etc.

**Golden Gate Claude worked because Anthropic found one feature that meant "Golden Gate Bridge" and turned its volume WAY up!**

In [ ]:
# Configure our SAE
sae_config = SAEConfig(
    d_in=model_config.n_embd,    # 128-dim input (matches model hidden size)
    expansion_factor=8,           # 1,024 features in our concept dictionary
    activation="topk",            # TopK: keep only the top-k most active features
    k=16,                         # Only 16 "concepts" active per thought-vector
    hook_point=hook_point,
    batch_size=256,
    num_epochs=8,
    learning_rate=3e-4,
)

print(f"SAE Architecture:")
print(f"  Input:     {sae_config.d_in} dimensions (model's thought-vector)")
print(f"  Features:  {sae_config.d_sae} (8x expansion = our concept dictionary)")
print(f"  Sparsity:  only {sae_config.k} of {sae_config.d_sae} features active at once")
print(f"\nAnalogy: We have a dictionary of {sae_config.d_sae} possible concepts,")
print(f"but each thought only uses {sae_config.k} of them — that's SPARSE!")

In [ ]:
# Split data into training and validation sets
n_val = 1500
train_acts = activations[n_val:]
val_acts = activations[:n_val]

# Create and train the SAE
sae = TopKSAE(sae_config)
trainer = SAETrainer(
    sae=sae,
    config=sae_config,
    activations=train_acts,
    val_activations=val_acts,
    device="cpu",  # SAE training is fast on CPU for this size
)

# Train!
print("Training SAE...\n")
t0 = time.time()
losses = []
for epoch in range(sae_config.num_epochs):
    metrics = trainer.train_epoch(verbose=False)
    losses.append(metrics["total_loss"])
    bar = "#" * int(30 * (epoch + 1) / sae_config.num_epochs)
    print(f"  [{bar:<30}] Epoch {epoch+1}/{sae_config.num_epochs}  "
          f"loss: {metrics['total_loss']:.6f}  L0: {metrics['l0']:.1f}")

elapsed = time.time() - t0
pct = (1 - losses[-1] / losses[0]) * 100
print(f"\nDone in {elapsed:.1f}s — loss reduced by {pct:.1f}%")

In [ ]:
# Plot training loss — the SAE is learning to reconstruct thoughts!
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(range(1, len(losses) + 1), losses, 'o-', color='#E24A33', linewidth=2, markersize=6)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Reconstruction Loss (MSE)', fontsize=12)
ax.set_title('SAE Training: Learning to Decompose Thoughts into Concepts', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, len(losses) + 1))
plt.tight_layout()
plt.show()
print("The SAE is learning to faithfully reconstruct thoughts using only sparse features!")

## Step 4: Find Our "Golden Gate" Features

Now let's evaluate the SAE and find the most interesting features. In the real experiment, Anthropic searched through millions of features to find one that corresponded to the Golden Gate Bridge. We'll pick our most active feature as our stand-in "Golden Gate."

> **Note:** Our model is randomly initialized (not trained on real text), so our features won't have obvious semantic meanings like "bridge" or "math." But the **mechanism is identical** — each feature is a direction in thought-space that the SAE has learned to isolate.

In [ ]:
# Evaluate SAE quality
evaluator = SAEEvaluator(sae, sae_config)
eval_metrics = evaluator.evaluate(val_acts, compute_dead_latents=True)

print("SAE Quality Report:")
print(f"  Explained Variance:  {eval_metrics.explained_variance * 100:.1f}%")
print(f"  Reconstruction MSE:  {eval_metrics.mse_loss:.6f}")
print(f"  Alive features:      {int((1 - eval_metrics.dead_latent_fraction) * sae_config.d_sae)}/{sae_config.d_sae}")
print(f"  L0 (active/thought): {eval_metrics.l0_mean:.1f}")

# Find the top features
viz = FeatureVisualizer(sae, sae_config)
top_indices, top_freqs = viz.get_top_features(val_acts, k=10)

print(f"\nTop 10 Most Active Features:")
print(f"{'Rank':<6} {'Feature':<10} {'Frequency':<12} {'Avg Strength':<15} {'Max Strength'}")
print("-" * 58)

for i, (idx, freq) in enumerate(zip(top_indices[:10], top_freqs[:10])):
    stats = viz.get_feature_statistics(idx.item(), val_acts)
    label = "  <-- OUR 'GOLDEN GATE'!" if i == 0 else ""
    print(f"{i+1:<6} #{idx.item():<9} {freq:.4f}       "
          f"{stats['mean_when_active']:<15.4f} {stats['max_activation']:.4f}{label}")

golden_feature = top_indices[0].item()
print(f"\nFeature #{golden_feature} selected as our 'Golden Gate Bridge' feature!")

In [ ]:
# Visualize feature frequencies — which features are most "popular"?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Feature frequency distribution
all_freqs = top_freqs[:10].numpy()
colors = ['#E8B130' if i == 0 else '#348ABD' for i in range(10)]
bars = axes[0].bar(range(10), all_freqs, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Feature Rank', fontsize=12)
axes[0].set_ylabel('Activation Frequency', fontsize=12)
axes[0].set_title('Top 10 Features by Frequency', fontsize=13)
axes[0].set_xticks(range(10))
axes[0].set_xticklabels([f'#{idx.item()}' for idx in top_indices[:10]], rotation=45, fontsize=9)
# Highlight Golden Gate feature
axes[0].annotate('Our "Golden\nGate" feature!', xy=(0, all_freqs[0]),
                 xytext=(2, all_freqs[0] * 1.15),
                 arrowprops=dict(arrowstyle='->', color='#E8B130', lw=2),
                 fontsize=11, color='#E8B130', fontweight='bold')

# Right: Alive vs Dead features (pie chart)
alive = int((1 - eval_metrics.dead_latent_fraction) * sae_config.d_sae)
dead = sae_config.d_sae - alive
axes[1].pie([alive, dead], labels=[f'Alive ({alive})', f'Dead ({dead})'],
            colors=['#2ECC71', '#E74C3C'], autopct='%1.0f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title(f'Feature Dictionary Health ({sae_config.d_sae} total)', fontsize=13)

plt.tight_layout()
plt.show()
print("Gold bar = our chosen 'Golden Gate' feature. Dead features never activate — room for improvement!")

## Step 5: THE GOLDEN GATE MOMENT — Feature Steering!

```
          _____
         /     \
    ____/       \____
   |  |    ( )    |  |
   |  |    | |    |  |
   |  |    | |    |  |
  _|  |____| |____|  |_
 / ________________________\
 \/\/\/\/\/\/\/\/\/\/\/\/\/\/
 ~~~~~~~~~~~~~~~~~~~~~~~~~~~
```

**This is the moment that made Golden Gate Claude famous!**

When Anthropic found the "Golden Gate Bridge" feature, they didn't just observe it — they **amplified** it during inference. They multiplied its activation by a large factor, effectively telling the model:

> *"Whatever you're thinking about, think about the Golden Gate Bridge MORE. A LOT more."*

The result? Claude started relating EVERYTHING to the bridge.

We'll do the same thing: take our chosen feature and crank it up to various strengths, watching how the model's output probability distribution changes.

In [ ]:
# Wrap our model with the InterpretableModel for steering
saes = {hook_point: sae}
interp_model = InterpretableModel(model, saes, device=device)

# Create a fixed input to compare steered vs unsteered
test_tokens = torch.randint(0, model_config.vocab_size, (1, 32), device=device)

# Get baseline (unsteered) output
with torch.no_grad():
    baseline_logits = interp_model(test_tokens)
baseline_probs = F.softmax(baseline_logits[:, -1, :], dim=-1)

# Now steer at increasing strengths!
strengths = [1.0, 2.0, 5.0, 10.0, 25.0, 50.0]
results = []

print(f"Steering feature #{golden_feature} at increasing strengths:\n")
print(f"{'Strength':<12} {'Top Token':<12} {'Top Prob':<12} {'Mean Shift':<15} {'Max Shift':<12} {'Verdict'}")
print("-" * 78)

for strength in strengths:
    with torch.no_grad():
        steered_logits = interp_model.steer(
            test_tokens,
            feature_id=(hook_point, golden_feature),
            strength=strength,
        )
    steered_probs = F.softmax(steered_logits[:, -1, :], dim=-1)
    diff = (steered_logits - baseline_logits).abs()
    top_tok = steered_probs.argmax().item()
    top_prob = steered_probs.max().item()
    mean_diff = diff.mean().item()
    max_diff = diff.max().item()
    changed = top_tok != baseline_probs.argmax().item()

    results.append({
        'strength': strength, 'top_tok': top_tok, 'top_prob': top_prob,
        'mean_diff': mean_diff, 'max_diff': max_diff, 'probs': steered_probs.cpu()
    })

    if strength <= 1.0:
        verdict = "baseline"
    elif strength <= 2.0:
        verdict = "gentle nudge"
    elif strength <= 5.0:
        verdict = "noticeable shift"
    elif strength <= 10.0:
        verdict = "STRONG shift!"
    else:
        verdict = "OBSESSED!"

    print(f"{strength:<12.1f} Token {top_tok:<6} {top_prob:<12.4f} {mean_diff:<15.4f} {max_diff:<12.4f} {verdict}")

In [ ]:
# Visualize how steering reshapes the output distribution
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f'How Feature #{golden_feature} Steering Reshapes Output Probabilities',
             fontsize=15, fontweight='bold')

# Plot probability distributions at each strength
top_n = 20  # Show top 20 tokens
baseline_top = baseline_probs[0, :top_n * 2].cpu()

for ax_idx, (result, ax) in enumerate(zip(results, axes.flatten())):
    strength = result['strength']
    probs = result['probs'][0]

    # Get top tokens from this distribution
    top_vals, top_idxs = torch.topk(probs, top_n)

    # Also show baseline for same tokens
    baseline_vals = baseline_probs[0, top_idxs].cpu()

    x = np.arange(top_n)
    width = 0.35

    ax.bar(x - width/2, baseline_vals.numpy(), width, label='Baseline',
           color='#348ABD', alpha=0.7)
    ax.bar(x + width/2, top_vals.numpy(), width, label=f'Steered ({strength}x)',
           color='#E8B130', alpha=0.85)

    ax.set_title(f'Strength = {strength}x', fontsize=12,
                 color='#E8B130' if strength >= 10 else 'black',
                 fontweight='bold' if strength >= 10 else 'normal')
    ax.set_ylabel('Probability', fontsize=10)
    ax.set_xlabel('Token rank', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, max(0.05, top_vals.max().item() * 1.2))

plt.tight_layout()
plt.show()

print("\nWatch how the gold bars (steered) diverge from blue bars (baseline) as strength increases!")
print("At high strengths, one token dominates — the model becomes 'obsessed' with one output.")

## Step 6: Feature Steering Playground

Let's explore more! We'll try:
1. **Different features** — each one steers behavior in a different direction
2. **Negative steering (suppression)** — the ANTI-Golden Gate! Push a feature DOWN instead of up
3. **Multi-feature steering** — combine multiple features at once

This is where it gets really fun. In a real trained model, these features would correspond to actual concepts — imagine having a "politeness dial", a "formality dial", and a "creativity dial" you could independently adjust!

In [ ]:
# --- Experiment 1: Different features steer in different directions ---
print("EXPERIMENT 1: Steering with different features (strength=10.0)\n")
print(f"{'Feature':<12} {'Top Token':<15} {'Top Prob':<12} {'Behavior Changed?'}")
print("-" * 55)

baseline_top_tok = baseline_probs.argmax().item()
feature_effects = []

for i, idx in enumerate(top_indices[:5]):
    feat_idx = idx.item()
    with torch.no_grad():
        steered = interp_model.steer(
            test_tokens, feature_id=(hook_point, feat_idx), strength=10.0
        )
    probs = F.softmax(steered[:, -1, :], dim=-1)
    top_tok = probs.argmax().item()
    top_prob = probs.max().item()
    changed = top_tok != baseline_top_tok

    label = " (our 'Golden Gate')" if i == 0 else ""
    print(f"#{feat_idx:<10} Token {top_tok:<9} {top_prob:<12.4f} {'YES!' if changed else 'same'}{label}")
    feature_effects.append({'feature': feat_idx, 'top_tok': top_tok, 'prob': top_prob, 'changed': changed})

print("\nEach feature pushes the model in a DIFFERENT direction!")
print("This is the power of SAE-based steering — precise, independent control.")

In [ ]:
# --- Experiment 2: Negative steering (suppression) ---
print("EXPERIMENT 2: The ANTI-Golden Gate — suppressing our feature\n")
print("What happens when we push our 'Golden Gate' feature NEGATIVE?\n")

suppress_strengths = [-1.0, -3.0, -5.0, -10.0]
print(f"{'Strength':<12} {'Top Token':<12} {'Top Prob':<12} {'Mean Shift'}")
print("-" * 48)

for strength in suppress_strengths:
    with torch.no_grad():
        suppressed = interp_model.steer(
            test_tokens, feature_id=(hook_point, golden_feature), strength=strength
        )
    probs = F.softmax(suppressed[:, -1, :], dim=-1)
    diff = (suppressed - baseline_logits).abs().mean().item()
    print(f"{strength:<12.1f} Token {probs.argmax().item():<6} {probs.max().item():<12.4f} {diff:.4f}")

print("\nNegative steering SUPPRESSES a concept — the opposite of amplification!")
print("In a real model, this could make it AVOID a topic entirely.")

In [ ]:
# --- Experiment 3: Visualize the steering "landscape" ---
print("EXPERIMENT 3: The Steering Landscape\n")
print("Sweeping from suppression (-10x) to amplification (+50x)...\n")

sweep_strengths = [-10, -5, -2, -1, 0, 1, 2, 5, 10, 25, 50]
sweep_probs = []
sweep_top_tokens = []
sweep_entropies = []

for strength in sweep_strengths:
    if strength == 0:
        logits = baseline_logits
    else:
        with torch.no_grad():
            logits = interp_model.steer(
                test_tokens, feature_id=(hook_point, golden_feature), strength=float(strength)
            )
    probs = F.softmax(logits[:, -1, :], dim=-1)
    entropy = -(probs * (probs + 1e-10).log()).sum().item()
    sweep_probs.append(probs.max().item())
    sweep_top_tokens.append(probs.argmax().item())
    sweep_entropies.append(entropy)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Top token probability vs steering strength
axes[0].plot(sweep_strengths, sweep_probs, 'o-', color='#E8B130', linewidth=2, markersize=8)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='No steering')
axes[0].axvspan(-10, 0, alpha=0.1, color='blue', label='Suppression zone')
axes[0].axvspan(0, 50, alpha=0.1, color='red', label='Amplification zone')
axes[0].set_xlabel('Steering Strength', fontsize=12)
axes[0].set_ylabel('Top Token Probability', fontsize=12)
axes[0].set_title('How Confident Is the Model?', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: Entropy (uncertainty) vs steering strength
axes[1].plot(sweep_strengths, sweep_entropies, 's-', color='#E24A33', linewidth=2, markersize=8)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Steering Strength', fontsize=12)
axes[1].set_ylabel('Output Entropy (bits)', fontsize=12)
axes[1].set_title('How Uncertain Is the Model?', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Left: High amplification -> model becomes very confident (one token dominates)")
print("Right: High amplification -> entropy drops (model becomes 'obsessed' with one answer)")
print("       This is exactly what happened with Golden Gate Claude!")

## Step 7: Under the Hood — Watching Features Fire in Real Time

Let's peek inside the model during inference. We'll track which features activate at each token position — like watching a real-time brain scan. This shows you **exactly** what the SAE is doing.

In [ ]:
# Track features during a forward pass
with interp_model.interpretation_enabled():
    with torch.no_grad():
        interp_model(test_tokens)
    features = interp_model.get_active_features()[hook_point]

print(f"Feature tensor shape: {features.shape}")
print(f"  (= {features.shape[0]} token positions x {features.shape[1]} possible features)")
print(f"  Active features per position: {(features != 0).sum(dim=-1).float().mean():.1f}")
print()

# Heatmap of feature activations across positions
n_pos = min(32, features.shape[0])
n_feat_show = 30  # Show top 30 most active features

# Find globally most active features
feat_activity = (features[:n_pos] != 0).float().sum(dim=0)
active_feat_indices = feat_activity.argsort(descending=True)[:n_feat_show]
heatmap_data = features[:n_pos, active_feat_indices].numpy()

fig, ax = plt.subplots(1, 1, figsize=(14, 6))
im = ax.imshow(heatmap_data.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xlabel('Token Position', fontsize=12)
ax.set_ylabel('Feature Index', fontsize=12)
ax.set_title(f'Feature Activations Across Token Positions (Top {n_feat_show} Features)', fontsize=13)
ax.set_yticks(range(n_feat_show))
ax.set_yticklabels([f'#{active_feat_indices[i].item()}' for i in range(n_feat_show)], fontsize=7)

# Highlight the Golden Gate feature if it's in the top
golden_gate_pos = (active_feat_indices == golden_feature).nonzero()
if len(golden_gate_pos) > 0:
    y_pos = golden_gate_pos[0].item()
    ax.annotate('"Golden Gate"', xy=(n_pos - 1, y_pos), xytext=(n_pos + 1, y_pos),
                fontsize=10, color='#E8B130', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#E8B130'))

plt.colorbar(im, ax=ax, label='Activation Strength')
plt.tight_layout()
plt.show()

print("Each row is a feature, each column is a token position.")
print("Bright spots = where a feature is strongly active.")
print("Sparse! Most cells are dark (zero) — only a few features fire per position.")

## What You Just Learned

Congratulations! You've recreated the core technique behind **Golden Gate Claude**. Here's what you now understand:

### The Big Ideas

| Concept | What It Means | Why It Matters |
|---------|--------------|----------------|
| **Sparse Autoencoders** | Decompose dense neural activations into sparse, interpretable features | We can "read" a model's thoughts |
| **Feature = Concept** | Each SAE feature corresponds to a direction in representation space | Individual concepts are isolatable |
| **Feature Steering** | Amplify/suppress features during inference to change behavior | Precise behavioral control without retraining |
| **Golden Gate Effect** | Extreme amplification makes the model "obsessed" with one concept | Demonstrates the power (and risk) of feature steering |

### The Golden Gate Claude Recipe
1. Train an SAE on your model's internal activations
2. Find a feature that corresponds to an interesting concept
3. During inference, multiply that feature's activation by a large number
4. Watch the model become obsessed!

### Next Steps

- **Train on real data**: Run `bash speedrun.sh` to train nanochat on actual text, then repeat this walkthrough with a model that has *real semantic features* — you might find your own "bridge" feature!
- **Try the full Colab notebook**: `colab_sae_training.ipynb` trains SAEs on the pre-trained nanochat-d32 model (1.88B params)
- **Explore deception features**: `colab_sae_deception_training.ipynb` uses Anthropic's datasets to find features related to deceptive behavior
- **Multi-layer analysis**: Train SAEs on different layers and compare — early layers capture syntax, late layers capture semantics
- **Read the paper**: Anthropic's [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) describes the full Golden Gate Claude experiment

### Why This Matters for AI Safety

Golden Gate Claude wasn't just a fun demo — it showed that we can understand and control AI behavior at a level deeper than prompting. This is crucial for:
- **Detecting** unwanted behaviors (deception, bias, harmful content)
- **Steering** models toward desired behaviors (helpfulness, honesty)
- **Understanding** what models actually learn vs. what we think they learn

You now have the tools to do this yourself. Happy feature hunting!